<a href="https://colab.research.google.com/github/SakshyamAryal077/Class-Balancing-Techniques-for-MIT-BIH-ARRYTHMIA-DETECTION./blob/main/wmcvaeipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# If needed, install imbalanced-learn
!pip install imbalanced-learn


In [ ]:
import numpy as np
import torch
from imblearn.over_sampling import ADASYN
from google.colab import files
from sklearn.preprocessing import LabelEncoder




In [ ]:
# Load your saved subset files
X_train_small = np.load("X_trainsmall.npy")
y_train_small = np.load("y_trainsmall.npy")

print("Original X_train_small shape:", X_train_small.shape)
print("Original y_train_small shape:", y_train_small.shape)

classes, counts = np.unique(y_train_small, return_counts=True)
print("\nClass distribution before adasyn:")
for cls, cnt in zip(classes, counts):
    print(f"Class {cls}: {cnt}")


Original X_train_small shape: (27010, 360, 1)
Original y_train_small shape: (27010,)

Class distribution before smoteenn:
Class 0: 193
Class 1: 21680
Class 2: 2734
Class 3: 667
Class 4: 1736


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WMCVAE(nn.Module):
    def __init__(self, input_dim, num_classes, latent_dim=32, hidden_dim=128):
        super(WMCVAE, self).__init__()

        self.input_dim = input_dim
        self.num_classes = num_classes
        self.latent_dim = latent_dim

        # Encoder
        self.encoder_fc1 = nn.Linear(input_dim + num_classes, hidden_dim)
        self.encoder_fc2 = nn.Linear(hidden_dim, hidden_dim)

        self.mu_layer = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.decoder_fc1 = nn.Linear(latent_dim + num_classes, hidden_dim)
        self.decoder_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def encode(self, x, y_onehot):
        h = torch.cat([x, y_onehot], dim=1)
        h = F.relu(self.encoder_fc1(h))
        h = F.relu(self.encoder_fc2(h))
        mu = self.mu_layer(h)
        logvar = self.logvar_layer(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y_onehot):
        h = torch.cat([z, y_onehot], dim=1)
        h = F.relu(self.decoder_fc1(h))
        h = F.relu(self.decoder_fc2(h))
        out = self.output_layer(h)
        return out

    def forward(self, x, y_onehot):
        mu, logvar = self.encode(x, y_onehot)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z, y_onehot)
        return recon_x, mu, logvar


In [ ]:
def wm_cvae_loss(recon_x, x, mu, logvar, class_weights=None, labels=None):
    recon_loss = F.mse_loss(recon_x, x, reduction='none').mean(dim=1)

    if class_weights is not None and labels is not None:
        sample_weights = class_weights[labels]
        recon_loss = recon_loss * sample_weights

    recon_loss = recon_loss.mean()

    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + kl_loss
    return total_loss, recon_loss, kl_loss


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

X_flat = X_train_small.reshape(X_train_small.shape[0], -1)
y_arr = y_train_small.copy()

X_tensor = torch.tensor(X_flat, dtype=torch.float32)
y_tensor = torch.tensor(y_arr, dtype=torch.long)

num_classes = len(np.unique(y_arr))
input_dim = X_flat.shape[1]

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_arr),
    y=y_arr
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

print("Input dim:", input_dim)
print("Num classes:", num_classes)
print("Class weights:", class_weights)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

wmcvae = WMCVAE(input_dim=input_dim, num_classes=num_classes, latent_dim=32, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(wmcvae.parameters(), lr=0.001)
class_weights = class_weights.to(device)

num_epochs = 20

for epoch in range(num_epochs):
    wmcvae.train()
    total_loss = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        y_onehot = F.one_hot(y_batch, num_classes=num_classes).float()

        optimizer.zero_grad()

        recon_x, mu, logvar = wmcvae(x_batch, y_onehot)
        loss, recon_loss, kl_loss = wm_cvae_loss(
            recon_x, x_batch, mu, logvar,
            class_weights=class_weights,
            labels=y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {total_loss/len(loader):.4f}")


In [ ]:
def generate_samples(model, class_label, num_samples, num_classes, latent_dim, device):
    model.eval()

    with torch.no_grad():
        z = torch.randn(num_samples, latent_dim).to(device)

        labels = torch.full((num_samples,), class_label, dtype=torch.long).to(device)
        y_onehot = F.one_hot(labels, num_classes=num_classes).float()

        generated = model.decode(z, y_onehot)
        return generated.cpu().numpy()


In [ ]:
import numpy as np
from collections import Counter

# Flatten original training data
X_original_flat = X_train_small.reshape(X_train_small.shape[0], -1)
y_original = y_train_small.copy()

class_counts = Counter(y_original)
max_count = max(class_counts.values())

X_generated_list = []
y_generated_list = []

for cls in sorted(class_counts.keys()):
    current_count = class_counts[cls]
    num_to_generate = max_count - current_count

    if num_to_generate > 0:
        generated_samples = generate_samples(
            model=wmcvae,
            class_label=cls,
            num_samples=num_to_generate,
            num_classes=num_classes,
            latent_dim=32,
            device=device
        )

        X_generated_list.append(generated_samples)
        y_generated_list.append(np.full(num_to_generate, cls))

# Combine generated samples
if len(X_generated_list) > 0:
    X_generated = np.vstack(X_generated_list)
    y_generated = np.hstack(y_generated_list)

    X_train_wmcvae_flat = np.vstack([X_original_flat, X_generated])
    y_train_wmcvae = np.hstack([y_original, y_generated])
else:
    X_train_wmcvae_flat = X_original_flat
    y_train_wmcvae = y_original

# Reshape back to original ECG shape
X_train_wmcvae = X_train_wmcvae_flat.reshape(-1, *X_train_small.shape[1:])

print("Final X_train_wmcvae shape:", X_train_wmcvae.shape)
print("Final y_train_wmcvae shape:", y_train_wmcvae.shape)

classes_after, counts_after = np.unique(y_train_wmcvae, return_counts=True)
print("\nClass distribution after WM-CVAE balancing:")
for cls, cnt in zip(classes_after, counts_after):
    print(f"Class {cls}: {cnt}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

classes_before, counts_before = np.unique(y_train_small, return_counts=True)
classes_after, counts_after = np.unique(y_train_wmcvae, return_counts=True)

plt.figure(figsize=(10, 6))

x = np.arange(len(classes_before))
width = 0.35

plt.bar(x - width/2, counts_before, width, label='Before', color='orange')
plt.bar(x + width/2, counts_after, width, label='After WM-CVAE', color='purple')

plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.title("Class Distribution Before and After WM-CVAE Balancing")
plt.xticks(x, classes_before)
plt.legend()
plt.show()


In [ ]:
import numpy as np

np.save("X_train_wmcvae.npy", X_train_wmcvae)
np.save("y_train_wmcvae.npy", y_train_wmcvae)

print("Saved files:")
print("X_train_wmcvae.npy")
print("y_train_wmcvae.npy")
